In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/updated_dataset.csv.gz', compression='gzip')

print("Shape:", df.shape)
print()
print(df['threat_label'].value_counts())
print()
print(df.head(3))

Shape: (1447167, 11)

threat_label
benign       964778
malicious    482389
Name: count, dtype: int64

             timestamp       source_ip        dest_ip protocol   action  \
0  2024-04-10T00:00:00   192.168.1.220  192.168.1.216      SSH  allowed   
1  2024-11-11T00:00:00   192.168.1.104   192.168.1.34      FTP  blocked   
2  2024-06-11T00:00:00  187.14.173.168  192.168.1.202    HTTPS  allowed   

  threat_label log_type  bytes_transferred             user_agent  \
0    malicious      ids              37110  Nmap Scripting Engine   
1       benign      ids              10727  Nmap Scripting Engine   
2    malicious      ids              46764  Nmap Scripting Engine   

               request_path  gatekeeper_label  
0          /home/user?hydra                 1  
1                    /files                 0  
2  /admin/config?backup.sql                 1  


In [ ]:
print(df["threat_label"].value_counts())

threat_label
benign       964778
malicious    482389
Name: count, dtype: int64


In [ ]:
# ====================================
# IMPROVED GATEKEEPER MODEL TRAINING (NO WARNINGS)
# ====================================

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_curve
from sklearn.preprocessing import LabelEncoder
import joblib
import warnings
warnings.filterwarnings('ignore')

# Load your data
df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/updated_dataset.csv.gz', compression='gzip')

# Prepare labels
df["threat_label"] = df["threat_label"].astype(str).str.lower().str.strip()
df["threat_label"] = df["threat_label"].map({"benign": 0, "malicious": 1})
df = df.dropna(subset=["threat_label"])
df["threat_label"] = df["threat_label"].astype(int)

print(f"Dataset shape: {df.shape}")
print(f"Malicious: {(df['threat_label']==1).sum()}, Benign: {(df['threat_label']==0).sum()}")

# ====================================
# ENHANCED FEATURE ENGINEERING
# ====================================

# Time features
df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')
df['hour'] = df['timestamp'].dt.hour.fillna(0)
df['is_night'] = ((df['hour'] < 6) | (df['hour'] > 22)).astype(int)

# Request path features (fixed to avoid warnings)
df['request_path'] = df['request_path'].astype(str)
df['request_depth'] = df['request_path'].apply(lambda x: min(x.count('/'), 10))

# Use simple string search instead of regex with groups
df['has_special_chars'] = df['request_path'].apply(
    lambda x: 1 if any(c in x for c in ['?', '=', '<', '>', '%', "'", '"', ';', '-']) else 0
)

df['has_sql_keywords'] = df['request_path'].str.lower().apply(
    lambda x: 1 if any(keyword in x for keyword in ['select', 'union', 'insert', 'drop', 'delete', 'exec']) else 0
)

df['has_path_traversal'] = df['request_path'].apply(
    lambda x: 1 if '../' in x or '..\\' in x else 0
)

df['num_digits'] = df['request_path'].str.count(r'\d').clip(0, 20)
df['path_length'] = df['request_path'].str.len().clip(0, 500)
df['is_long_request'] = (df['path_length'] > 100).astype(int)

# Bytes features
df['bytes_log'] = np.log1p(df['bytes_transferred'])
df['is_large_transfer'] = (df['bytes_transferred'] > 50000).astype(int)

# Encode categorical features
le_protocol = LabelEncoder()
le_action = LabelEncoder()
le_log_type = LabelEncoder()

df['protocol_encoded'] = le_protocol.fit_transform(df['protocol'].astype(str))
df['action_encoded'] = le_action.fit_transform(df['action'].astype(str))
df['log_type_encoded'] = le_log_type.fit_transform(df['log_type'].astype(str))

# User agent features
df['user_agent'] = df['user_agent'].astype(str).str.lower()
df['has_nmap'] = df['user_agent'].apply(lambda x: 1 if 'nmap' in x else 0)
df['has_curl'] = df['user_agent'].apply(lambda x: 1 if 'curl' in x else 0)
df['ua_length'] = df['user_agent'].str.len().clip(0, 200)

# ====================================
# FEATURE SELECTION
# ====================================
FEATURES = [
    'action_encoded',
    'log_type_encoded',
    'has_special_chars',
    'has_sql_keywords',
    'has_path_traversal',
    'num_digits',
    'is_long_request',
    'request_depth',
    'path_length',
    'protocol_encoded',
    'bytes_log',
    'is_large_transfer',
    'hour',
    'is_night',
    'has_nmap',
    'has_curl',
    'ua_length'
]

X = df[FEATURES].fillna(0)
y = df['threat_label']

print(f"Features shape: {X.shape}")

# ====================================
# TRAIN WITH OPTIMIZED PARAMETERS
# ====================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")
print(f"Train malicious %: {y_train.mean()*100:.1f}%")

# Optimized Random Forest
model = RandomForestClassifier(
    n_estimators=500,
    max_depth=15,
    min_samples_split=5,
    min_samples_leaf=2,
    max_features='sqrt',
    class_weight='balanced',
    random_state=42,
    n_jobs=-1,
    verbose=1
)

print("\nTraining Random Forest...")
model.fit(X_train, y_train)

# ====================================
# EVALUATE
# ====================================
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print("\n" + "="*60)
print("MODEL EVALUATION")
print("="*60)

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Benign', 'Malicious']))

# Find optimal threshold
precisions, recalls, thresholds = precision_recall_curve(y_test, y_proba)
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-9)
best_idx = np.argmax(f1_scores[:-1])
best_threshold = thresholds[best_idx]

print(f"\nOptimal threshold: {best_threshold:.4f}")

# Apply threshold
y_pred_optimized = (y_proba >= best_threshold).astype(int)

# Confusion matrix
cm = confusion_matrix(y_test, y_pred_optimized)
tn, fp, fn, tp = cm.ravel()

print("\n" + "="*60)
print("CONFUSION MATRIX (Optimized)")
print("="*60)
print(f"True Negatives (Correct Benign)      : {tn:,}")
print(f"False Positives (Benign as Malicious): {fp:,}")
print(f"False Negatives (Malicious as Benign): {fn:,}")
print(f"True Positives (Correct Malicious)   : {tp:,}")

# Calculate metrics
accuracy = (tp + tn) / (tp + tn + fp + fn)
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

print("\n" + "="*60)
print("PERFORMANCE METRICS")
print("="*60)
print(f"Accuracy  : {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"Precision : {precision:.4f} ({precision*100:.2f}%)")
print(f"Recall    : {recall:.4f} ({recall*100:.2f}%)")
print(f"F1 Score  : {f1:.4f}")
print(f"False Positives: {fp}")
print(f"False Negatives: {fn}")

# ====================================
# SAVE IMPROVED MODEL
# ====================================
print("\n" + "="*60)
print("SAVING MODEL")
print("="*60)

joblib.dump(model, '/content/drive/MyDrive/Colab Notebooks/application_gatekeeper_streaming_improved.pkl')
joblib.dump(FEATURES, '/content/drive/MyDrive/Colab Notebooks/feature_app_columns_improved.pkl')
joblib.dump(best_threshold, '/content/drive/MyDrive/Colab Notebooks/app_threshold_improved.pkl')
joblib.dump(le_protocol, '/content/drive/MyDrive/Colab Notebooks/le_protocol.pkl')
joblib.dump(le_action, '/content/drive/MyDrive/Colab Notebooks/le_action.pkl')
joblib.dump(le_log_type, '/content/drive/MyDrive/Colab Notebooks/le_log_type.pkl')

print("✅ Model saved to: application_gatekeeper_streaming_improved.pkl")
print("✅ Features saved to: feature_app_columns_improved.pkl")
print("✅ Threshold saved to: app_threshold_improved.pkl")

# ====================================
# FEATURE IMPORTANCE
# ====================================
importance_df = pd.DataFrame({
    'feature': FEATURES,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print("\n" + "="*60)
print("TOP 10 FEATURE IMPORTANCES")
print("="*60)
for i, row in importance_df.head(10).iterrows():
    print(f"  {row['feature']:25s}: {row['importance']:.4f}")

print("\n✅ Training complete! Use this improved model in your dashboard.")

Dataset shape: (1447167, 11)
Malicious: 482389, Benign: 964778
Features shape: (1447167, 17)
Train shape: (1157733, 17), Test shape: (289434, 17)
Train malicious %: 33.3%

Training Random Forest...


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=-1)]: Done  46 tasks      | elapsed:   30.1s
[Parallel(n_jobs=-1)]: Done 196 tasks      | elapsed:  2.0min
[Parallel(n_jobs=-1)]: Done 446 tasks      | elapsed:  4.8min
[Parallel(n_jobs=-1)]: Done 500 out of 500 | elapsed:  5.5min finished
[Parallel(n_jobs=2)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done  46 tasks      | elapsed:    0.4s
[Parallel(n_jobs=2)]: Done 196 tasks      | elapsed:    1.7s
[Parallel(n_jobs=2)]: Done 446 tasks      | elapsed:    4.1s
[Parallel(n_jobs=2)]: Done 500 out of 500 | elapsed:    4.6s finished
[Parallel(n_jobs=2)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done  46 tasks      | elapsed:    0.4s
[Parallel(n_jobs=2)]: Done 196 tasks      | elapsed:    1.8s
[Parallel(n_jobs=2)]: Done 446 tasks      | elapsed:    4.2s
[Parallel(n_jobs=2)]: Done 500 out of 500 | elapsed:    5.1s finis


MODEL EVALUATION

Classification Report:
              precision    recall  f1-score   support

      Benign       1.00      1.00      1.00    192956
   Malicious       1.00      1.00      1.00     96478

    accuracy                           1.00    289434
   macro avg       1.00      1.00      1.00    289434
weighted avg       1.00      1.00      1.00    289434


Optimal threshold: 0.8656

CONFUSION MATRIX (Optimized)
True Negatives (Correct Benign)      : 192,956
False Positives (Benign as Malicious): 0
False Negatives (Malicious as Benign): 187
True Positives (Correct Malicious)   : 96,291

PERFORMANCE METRICS
Accuracy  : 0.9994 (99.94%)
Precision : 1.0000 (100.00%)
Recall    : 0.9981 (99.81%)
F1 Score  : 0.9990
False Positives: 0
False Negatives: 187

SAVING MODEL
✅ Model saved to: application_gatekeeper_streaming_improved.pkl
✅ Features saved to: feature_app_columns_improved.pkl
✅ Threshold saved to: app_threshold_improved.pkl

TOP 10 FEATURE IMPORTANCES
  has_special_chars    

In [4]:
# Check unique values in your dataset
print(f"Unique log_type_encoded values: {df['log_type_encoded'].unique()}")
# Output: [2]  ← Only one value, no predictive power

Unique log_type_encoded values: [2 1 0]


In [5]:
# Check if log_type column existed in raw data
if 'log_type' in df.columns:
    print(f"Raw log_type values: {df['log_type'].unique()}")
else:
    print("log_type column missing from raw data!")
    # Then log_type_encoded will always be 0

Raw log_type values: ['ids' 'firewall' 'application']
